# ORCID × arXiv Publication Matching

How can researcher records be connected to preprints?

Normalize nested researcher records, construct people, works and employment tables, compare CSV and Parquet exports, and link works to preprints.


In [ ]:
from pathlib import Path

Path("tables").mkdir(exist_ok=True)

In [ ]:
import gzip
import json

import pandas as pd

records = json.load(gzip.open("data/orcid_records.json.gz"))

len(records)

In [ ]:
pd.DataFrame(records).head()

In [ ]:
records[0].keys()

In [ ]:
people = pd.json_normalize(records)

people = people.rename(columns={
    "orcid-identifier.path": "orcid_id",
    "person.name.given-names.value": "given",
    "person.name.family-name.value": "family"
})

people["country"] = [
    record["person"]["addresses"]["address"][0]["country"]["value"]
    if record["person"]["addresses"]["address"] else None
    for record in records
]

people["n_works"] = [
    len(record["activities-summary"]["works"]["group"])
    for record in records
]

people["n_employments"] = [
    len(record["activities-summary"]["employments"]["affiliation-group"])
    for record in records
]

people = people[[
    "orcid_id", "given", "family", "country", "n_works", "n_employments"
]]

In [ ]:
people.head()

In [ ]:
work_rows = []
n_claims = 0

for record in records:
    for group in record["activities-summary"]["works"]["group"]:
        n_claims += len(group["work-summary"])

        claim = group["work-summary"][0].copy()
        claim["orcid_id"] = record["orcid-identifier"]["path"]

        identifiers = (
            claim["external-ids"]["external-id"]
            if claim["external-ids"] else []
        )

        dois = [
            item["external-id-value"]
            for item in identifiers
            if item["external-id-type"] == "doi"
        ]

        arxiv_ids = [
            item["external-id-value"]
            for item in identifiers
            if item["external-id-type"] == "arxiv"
        ]

        claim["doi"] = dois[0] if dois else None
        claim["arxiv_id"] = arxiv_ids[0] if arxiv_ids else None

        work_rows.append(claim)

works = pd.json_normalize(work_rows)

works = works.rename(columns={
    "title.title.value": "title",
    "publication-date.year.value": "year"
})

works["year"] = works["year"].astype("Int64")

works = works[["orcid_id", "title", "type", "year", "doi", "arxiv_id"]]

n_groups = len(works)

In [ ]:
works.head()

In [ ]:
type_shares = works["type"].value_counts() / len(works) * 100
type_shares.head().round(2)

In [ ]:
employment_rows = []

for record in records:
    for group in record["activities-summary"]["employments"]["affiliation-group"]:
        claim = group["summaries"][0]["employment-summary"].copy()
        claim["orcid_id"] = record["orcid-identifier"]["path"]
        employment_rows.append(claim)

employments = pd.json_normalize(employment_rows)

employments = employments.rename(columns={
    "organization.name": "org",
    "organization.address.country": "org_country",
    "role-title": "role",
    "start-date.year.value": "start_year",
    "end-date.year.value": "end_year"
})

employments["start_year"] = employments["start_year"].astype("Int64")
employments["end_year"] = employments["end_year"].astype("Int64")

employments = employments[[
    "orcid_id", "org", "org_country", "role", "start_year", "end_year"
]]

In [ ]:
employments.head()

In [ ]:
works.to_csv("tables/works.csv", index=False)
works.to_parquet("tables/works.parquet", index=False)

works_csv = pd.read_csv("tables/works.csv")
works_parquet = pd.read_parquet("tables/works.parquet")

pd.DataFrame({
    "original": works.dtypes,
    "csv": works_csv.dtypes,
    "parquet": works_parquet.dtypes
})

In [ ]:
years = list(range(2010, int(works["year"].max()) + 1))
works_by_year = people[["orcid_id"]].copy()

for year in years:
    counts = works[works["year"] == year].groupby("orcid_id").agg(
        n_works=("orcid_id", "count")
    ).reset_index()

    counts = counts.rename(columns={"n_works": year})
    works_by_year = works_by_year.merge(counts, on="orcid_id", how="left")
    works_by_year[year] = works_by_year[year].fillna(0).astype(int)


works_per_year = works_by_year[years].sum()
print("Missing year:", works["year"].isna().sum())
works_per_year

In [ ]:
arxiv = pd.read_csv("data/arxiv_econ_se.csv.gz", dtype={"id": str})
arxiv = arxiv.rename(columns={"id": "matched_id"})
preprints = arxiv[["matched_id", "categories"]]

works_clean = works.copy()
works_clean["work_id"] = works_clean.index

unmatched = works[
    works["arxiv_id"].notna() &
    ~works["arxiv_id"].isin(arxiv["matched_id"])
]
print(unmatched[["arxiv_id", "doi"]].head(15))

source = works_clean[["work_id", "arxiv_id"]].rename(
    columns={"arxiv_id": "matched_id"}
)
matched_links = source.merge(
    preprints, on="matched_id", how="inner", validate="many_to_one"
)
print("Before fixes:", matched_links["work_id"].nunique())

source["matched_id"] = works_clean["arxiv_id"].str.extract(
    r"(\d{4}\.\d{4,5})", expand=False
)
new_matches = source.merge(
    preprints, on="matched_id", how="inner", validate="many_to_one"
)
matched_links = pd.concat([matched_links, new_matches]).drop_duplicates(
    ["work_id", "matched_id"]
)
print("After cleaning arxiv_id:", matched_links["work_id"].nunique())

arxiv_pattern = r"(?i)(?:^|arxiv[:.]|/abs/|/pdf/)(\d{4}\.\d{4,5})"

print(works[works["doi"].str.contains("arxiv", case=False, na=False)]["doi"].head())

source["matched_id"] = works_clean["doi"].str.extract(
    arxiv_pattern, expand=False
)
new_matches = source.merge(
    preprints, on="matched_id", how="inner", validate="many_to_one"
)
matched_links = pd.concat([matched_links, new_matches]).drop_duplicates(
    ["work_id", "matched_id"]
)
print("After arXiv IDs in DOI:", matched_links["work_id"].nunique())

doi_pattern = r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)"

works_clean["doi_clean"] = (
    works_clean["doi"].str.strip().str.lower()
    .str.replace(doi_pattern, "", regex=True)
)
arxiv["doi_clean"] = (
    arxiv["doi"].str.strip().str.lower()
    .str.replace(doi_pattern, "", regex=True)
)

doi_lookup = arxiv[["doi_clean", "matched_id", "categories"]].dropna(
    subset=["doi_clean"]
)
new_matches = works_clean[["work_id", "doi_clean"]].dropna(
    subset=["doi_clean"]
).merge(doi_lookup, on="doi_clean", how="inner")

matched_links = pd.concat([
    matched_links,
    new_matches[["work_id", "matched_id", "categories"]]
]).drop_duplicates(["work_id", "matched_id"])

print("After journal DOI:", matched_links["work_id"].nunique())

identifier_rows = []
work_id = 0

for record in records:
    for group in record["activities-summary"]["works"]["group"]:
        for claim in group["work-summary"]:
            identifiers = (
                claim["external-ids"]["external-id"]
                if claim["external-ids"] else []
            )
            for item in identifiers:
                if (
                    item["external-id-type"] in ["doi", "arxiv"]
                    and item["external-id-relationship"] == "self"
                ):
                    identifier_rows.append({
                        "work_id": work_id,
                        "id_type": item["external-id-type"],
                        "raw_id": item["external-id-value"]
                    })
        work_id += 1

additional_ids = pd.DataFrame(identifier_rows)

additional_ids["matched_id"] = additional_ids["raw_id"].str.extract(
    arxiv_pattern, expand=False
)
new_matches = additional_ids[["work_id", "matched_id"]].merge(
    preprints, on="matched_id", how="inner", validate="many_to_one"
)

additional_ids["doi_clean"] = (
    additional_ids["raw_id"].str.strip().str.lower()
    .str.replace(doi_pattern, "", regex=True)
)
extra_dois = additional_ids[
    additional_ids["id_type"] == "doi"
][["work_id", "doi_clean"]].dropna(subset=["doi_clean"]).merge(
    doi_lookup, on="doi_clean", how="inner"
)

matched_links = pd.concat([
    matched_links,
    new_matches,
    extra_dois[["work_id", "matched_id", "categories"]]
]).drop_duplicates(["work_id", "matched_id"])

print("After additional self IDs:", matched_links["work_id"].nunique())

matched = works_clean.merge(
    matched_links, on="work_id", how="inner", validate="one_to_many"
)

print("Matched researchers:", matched["orcid_id"].nunique())

In [ ]:
field_matches = matched[
    ["orcid_id", "matched_id", "categories"]
].drop_duplicates(["orcid_id", "matched_id"]).copy()

field_matches["field"] = [
    "software engineering" if category.split()[0] == "cs.SE"
    else "economics"
    for category in field_matches["categories"]
]

field_counts = field_matches.groupby(["orcid_id", "field"]).agg(
    n_preprints=("matched_id", "count")
).reset_index()

researcher_fields = field_counts.sort_values(
    ["orcid_id", "n_preprints", "field"],
    ascending=[True, False, True]
).drop_duplicates("orcid_id")

researcher_fields = researcher_fields[["orcid_id", "field"]]

people_with_field = people.merge(
    researcher_fields, on="orcid_id",
    how="left", validate="one_to_one"
)

works_with_field = works.merge(
    researcher_fields, on="orcid_id",
    how="left", validate="many_to_one"
)

econ_works = works_with_field[
    works_with_field["field"] == "economics"
]
se_works = works_with_field[
    works_with_field["field"] == "software engineering"
]

types_by_field = pd.DataFrame({
    "economics": econ_works["type"].value_counts() / len(econ_works),
    "software engineering": se_works["type"].value_counts() / len(se_works)
}).fillna(0)

print("Researchers matching both fields:",
      (field_counts["orcid_id"].value_counts() > 1).sum())
print("Researchers without a field:",
      people_with_field["field"].isna().sum())
print(researcher_fields["field"].value_counts())

types_by_field.to_csv("tables/types_by_field.csv")
researcher_fields.to_csv("tables/researcher_fields.csv", index=False)

types_by_field.round(4)

In [ ]:
import re

author_data = pd.read_csv(
    "data/arxiv_econ_se.csv.gz", dtype={"id": str}
).rename(columns={"id": "matched_id"})

linked_authors = matched[
    ["orcid_id", "matched_id"]
].drop_duplicates().merge(
    author_data[["matched_id", "authors"]],
    on="matched_id",
    validate="many_to_one"
)

author_rows = []

for _, person in people.dropna(subset=["given", "family"]).iterrows():
    name = (person["given"] + " " + person["family"]).strip()
    papers = linked_authors[
        linked_authors["orcid_id"] == person["orcid_id"]
    ]

    for _, paper in papers.iterrows():
        found = re.search(
            r"(?<!\w)" + re.escape(name) + r"(?!\w)",
            paper["authors"],
            re.I
        )

        if found:
            rest = paper["authors"][found.end():].lstrip()

            if rest.startswith("("):
                variant = "with brackets"
            elif not rest or rest.startswith(",") or rest.startswith("and "):
                variant = "without brackets"
            else:
                continue

            author_rows.append({
                "orcid_id": person["orcid_id"],
                "name": name,
                "matched_id": paper["matched_id"],
                "variant": variant,
                "authors": paper["authors"]
            })

author_examples = pd.DataFrame(author_rows)

variant_counts = author_examples.groupby(
    ["orcid_id", "name"]
)["variant"].nunique().reset_index(name="n_variants")

confirmed_authors = variant_counts[
    variant_counts["n_variants"] == 2
]

explore = author_examples.merge(
    confirmed_authors[["orcid_id", "name"]],
    on=["orcid_id", "name"],
    validate="many_to_one"
).sort_values(["name", "variant", "matched_id"])

explore.to_csv("tables/explore_author_variants.csv", index=False)

print("Researchers with both variants:", len(confirmed_authors))
print(confirmed_authors[["name", "orcid_id"]].to_string(index=False))
print(explore.to_string(index=False))

In [ ]:
check_people_rows = []
check_work_rows = []
check_employment_rows = []
check_claims = 0

for record in records:
    oid = record["orcid-identifier"]["path"]
    person = record["person"]
    name = person.get("name") or {}
    addresses = (person.get("addresses") or {}).get("address") or []
    activities = record["activities-summary"]
    work_groups = activities["works"].get("group") or []
    employment_groups = activities["employments"].get("affiliation-group") or []

    check_people_rows.append({
        "orcid_id": oid,
        "given": (name.get("given-names") or {}).get("value"),
        "family": (name.get("family-name") or {}).get("value"),
        "country": addresses[0]["country"]["value"] if addresses else None,
        "n_works": len(work_groups),
        "n_employments": len(employment_groups)
    })

    for group in work_groups:
        claims = group["work-summary"]
        check_claims += len(claims)
        claim = claims[0]
        identifiers = (claim.get("external-ids") or {}).get("external-id") or []
        identifiers_by_type = {}

        for identifier in identifiers:
            key = identifier["external-id-type"]
            if key not in identifiers_by_type:
                identifiers_by_type[key] = identifier["external-id-value"]

        date = claim.get("publication-date") or {}

        check_work_rows.append({
            "orcid_id": oid,
            "title": ((claim.get("title") or {}).get("title") or {}).get("value"),
            "type": claim["type"],
            "year": (date.get("year") or {}).get("value"),
            "doi": identifiers_by_type.get("doi"),
            "arxiv_id": identifiers_by_type.get("arxiv")
        })

    for group in employment_groups:
        employment = group["summaries"][0]["employment-summary"]
        organization = employment.get("organization") or {}

        check_employment_rows.append({
            "orcid_id": oid,
            "org": organization.get("name"),
            "org_country": (organization.get("address") or {}).get("country"),
            "role": employment.get("role-title"),
            "start_year": (
                (employment.get("start-date") or {}).get("year") or {}
            ).get("value"),
            "end_year": (
                (employment.get("end-date") or {}).get("year") or {}
            ).get("value")
        })

check_people = pd.DataFrame(check_people_rows)
check_works = pd.DataFrame(check_work_rows)
check_employments = pd.DataFrame(check_employment_rows)

check_works["year"] = pd.to_numeric(check_works["year"]).astype("Int64")
for column in ["start_year", "end_year"]:
    check_employments[column] = pd.to_numeric(
        check_employments[column]
    ).astype("Int64")

checks = {}

for label, actual, rebuilt in [
    ("1.1: people", people, check_people),
    ("1.2: works", works, check_works),
    ("1.3: employments", employments, check_employments)
]:
    try:
        pd.testing.assert_frame_equal(
            actual.reset_index(drop=True),
            rebuilt[actual.columns].reset_index(drop=True),
            check_dtype=False
        )
        checks[label] = True
    except AssertionError as error:
        checks[label] = False
        print(label, str(error)[:500])

checks["1.2: claim count"] = n_claims == check_claims
checks["1.2: group count"] = n_groups == len(check_works)

check_yearly = (
    check_works[check_works["year"] >= 2010]
    .groupby(["orcid_id", "year"]).size().unstack(fill_value=0)
    .reindex(
        index=check_people["orcid_id"],
        columns=range(2010, int(check_works["year"].max()) + 1),
        fill_value=0
    )
)

try:
    pd.testing.assert_frame_equal(
        works_by_year.set_index("orcid_id").sort_index(),
        check_yearly.sort_index(),
        check_dtype=False,
        check_names=False
    )
    checks["2.2: every researcher-year count"] = True
except AssertionError as error:
    checks["2.2: every researcher-year count"] = False
    print("2.2:", str(error)[:500])

checks["2.1: Parquet preserves data and types"] = works.equals(works_parquet)
checks["2.1: CSV preserves years"] = works["year"].equals(
    works_csv["year"].astype("Int64")
)

check_field_works = check_works.merge(
    researcher_fields, on="orcid_id", how="left", validate="many_to_one"
)

check_type_counts = (
    check_field_works.groupby(["type", "field"]).size().unstack(fill_value=0)
)
check_shares = check_type_counts / check_type_counts.sum()

try:
    pd.testing.assert_frame_equal(
        types_by_field.sort_index().sort_index(axis=1),
        check_shares.sort_index().sort_index(axis=1),
        check_dtype=False,
        check_names=False
    )
    checks["3.2: all type shares"] = True
except AssertionError as error:
    checks["3.2: all type shares"] = False
    print("3.2:", str(error)[:500])

pd.DataFrame.from_dict(checks, orient="index", columns=["Passed"])

In [ ]:
people.to_csv("tables/people.csv", index=False)
employments.to_csv("tables/employments.csv", index=False)

pd.DataFrame({
    "measure": ["work groups", "claims"],
    "count": [n_groups, n_claims]
}).to_csv("tables/work_counts.csv", index=False)

pd.DataFrame({
    "stage": ["before fixes", "after fixes"],
    "matched_works": [20, matched["work_id"].nunique()]
}).to_csv("tables/match_counts.csv", index=False)

field_counts.to_csv("tables/field_counts.csv", index=False)
researcher_fields.to_csv("tables/researcher_fields.csv", index=False)
explore.to_csv("tables/explore_author_variants.csv", index=False)